<a href="https://colab.research.google.com/github/AnIsAsPe/ClasificadorClientesTelecom/blob/master/notebooks/pandas_30min_polizas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas en 30 minutos: cartera de pólizas
### Series, DataFrames, `loc` / `iloc` y filtrado booleano



In [ ]:
import numpy as np
import pandas as pd
print('pandas', pd.__version__)

---
## 1. Series: mitad arreglo de NumPy, mitad diccionario



### 1a. Como arreglo: operaciones vectorizadas, sin ciclos

In [ ]:
primas = np.array([12000, 9500, 28000, 15000, 42000])
print(primas *5)                    # ajuste del 5 % a toda la cartera sin ciclos
print(f'Prima total: {primas.sum()}')

In [ ]:
primas = np.array([12000, 9500, 28000, 15000, 42000])
s = pd.Series(primas)
s

In [ ]:
# Predicción:
print(s * 1.05)
print('Prima total:', s.sum())

### 1b. Como diccionario: acceso por etiqueta

In [ ]:
s = pd.Series({'P101': 12000, 'P102': 9500, 'P103': 28000, 'P104': 15000, 'P105': 42000})
s

In [ ]:
# Predicción:
print(s['P103'])        # acceso por "llave"
print('P101' in s)      # ¿revisa valores o etiquetas?
print(12000 in s)       # ¿y esto?
print(type(s.values))   # por dentro, un arreglo de NumPy

### 1c. Lo que sale de juntar ambas ideas: **alineación por etiqueta**
Al operar dos Series, pandas empareja por etiqueta, no por posición (parecido a un `JOIN` por llave).

In [ ]:
ajustes = pd.Series({'P105': 1000, 'P101': 500})   # otro orden y menos pólizas
# Predicción: ¿qué pasa con P102, P103 y P104?
print(s + ajustes)
print(s.add(ajustes, fill_value=0))

> **Desde R:** en un vector con nombres, `c(a=1, b=2) + c(b=10, a=20)` suma **por posición**. pandas suma **por etiqueta**.
> **Desde SQL:** `s + ajustes` se comporta como un `FULL OUTER JOIN` por llave; `fill_value=0` es como un `COALESCE(x, 0)`.

### Ejercicio
1. Muestra la prima de la póliza `P104`.
2. Crea `primas_2027` con un aumento del 4 %.
3. Muestra la prima más alta usando un método de la Series.

In [ ]:
# 1
print(___)

# 2
primas_2027 = ___
print(primas_2027)

# 3
print(___)

In [ ]:
#@title Ver solución (doble clic para abrir)
print(s['P104'])
primas_2027 = s * 1.04
print(primas_2027)
print(s.max())

---
## 2. DataFrame: un diccionario de Series que comparten índice

In [ ]:
datos = {
    'poliza':     ['P101', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108'],
    'ramo':       ['Auto', 'Vida', 'GMM', 'Auto', 'GMM', 'Vida', 'Auto', 'GMM'],
    'estado':     ['CDMX', 'Jalisco', 'Guanajuato', 'Nuevo León', 'CDMX', 'Guanajuato', 'Guanajuato', 'Jalisco'],
    'edad':       [34, 52, 45, 27, 61, 38, 41, 29],
    'prima':      [12000, 9500, 28000, 15000, 42000, 7000, 11000, 22000],
    'siniestros': [1, 0, 2, 3, 1, 0, 1, 2],
    'monto_sin':  [8000, 0, 35000, 21000, 18000, 0, 4500, 30500],
}
df = pd.DataFrame(datos)
df

In [ ]:
# Predicción: ¿de qué tipo es una columna?
print(type(df['prima']))
print(df.dtypes)

In [ ]:
# Nueva columna = nueva "llave"; el cálculo es vectorizado
df['siniestralidad'] = df['monto_sin'] / df['prima']
df

Hacemos que el número de póliza sea la **etiqueta de las filas** (el índice):

In [ ]:
if 'poliza' in df.columns:      # así la celda se puede ejecutar varias veces sin error
    df = df.set_index('poliza')
df

> **Equivalencias**
>
> | pandas | R (dplyr / base) | SQL |
> |---|---|---|
> | `df['prima']` | `df$prima` | `SELECT prima` |
> | `df[['ramo', 'prima']]` | `select(df, ramo, prima)` | `SELECT ramo, prima` |
> | `df['x'] = a / b` | `mutate(df, x = a / b)` | `SELECT *, a / b AS x` |
> | `df.set_index('poliza')` | `rownames(df) <- df$poliza` | `PRIMARY KEY (poliza)` |

### Ejercicio
1. Muestra cuántas filas y columnas tiene `df` (pista: es un atributo, no un método).
2. Agrega la columna `tuvo_siniestro`: `True` si `siniestros` es mayor que 0.

In [ ]:
# 1
print(df.___)

# 2
df['tuvo_siniestro'] = ___
df

In [ ]:
#@title Ver solución (doble clic para abrir)
print(df.shape)
df['tuvo_siniestro'] = df['siniestros'] > 0
df

---
## 3. `loc` e `iloc`

| | Selecciona por… | Ejemplo |
|---|---|---|
| `loc` | **etiquetas** | `df.loc['P103', 'prima']` |
| `iloc` | **posiciones** (empiezan en 0) | `df.iloc[2, 3]` |

Forma general: `df.loc[filas, columnas]` y `df.iloc[filas, columnas]`.

In [ ]:
df.loc['P103']

In [ ]:
df.loc['P103', 'siniestralidad']

In [ ]:
# Predicción: ¿cuántas filas salen? ¿aparece P105?
df.loc['P102':'P105', ['ramo', 'prima']]

In [ ]:
df.iloc[2]

In [ ]:
# Predicción: ¿cuántas filas? ¿qué columnas?
df.iloc[1:4, 0:2]

In [ ]:
df.iloc[-1]       # la última fila, igual que en las listas de Python

> **Clave:** con `loc` el rebanado **incluye** el final (`'P102':'P105'` trae 4 pólizas). Con `iloc` lo **excluye** (`1:4` trae 3).
>
> **Desde R:** `df[3, ]` en R equivale a `df.iloc[2]` en pandas, porque Python cuenta desde 0. El `2:4` de R incluye el final, como `loc`; el `1:4` de `iloc` no.
> **Desde SQL:** `loc` es como `WHERE poliza = 'P103'`. `iloc` no tiene equivalente directo, porque una tabla SQL no tiene orden; lo más parecido es `ORDER BY ... LIMIT ... OFFSET`.

### La trampa del índice numérico
Muy común cuando los identificadores son enteros:

In [ ]:
t = pd.Series([100, 200, 300], index=[2, 0, 1])
# Predicción: ¿dan lo mismo?
print(t.loc[0], t.iloc[0])

### Ejercicio
1. Con `loc`: el ramo y el estado de `P105`.
2. Con `iloc`: las tres primeras pólizas y sus dos primeras columnas.
3. Con `loc`: la siniestralidad de `P101`, `P105` y `P108` (pista: una **lista** de etiquetas).

In [ ]:
# 1
print(df.loc[___])

# 2
print(df.iloc[___])

# 3
print(df.loc[___])

In [ ]:
#@title Ver solución (doble clic para abrir)
print(df.loc['P105', ['ramo', 'estado']])
print(df.iloc[:3, :2])
print(df.loc[['P101', 'P105', 'P108'], 'siniestralidad'])

---
## 4. Filtrado booleano

Una comparación sobre una columna devuelve una **Series de `True`/`False`** (una *máscara*). Dentro de `df[...]` o `df.loc[...]` solo deja pasar las filas con `True`.

In [ ]:
mascara = df['siniestralidad'] > 1
mascara

In [ ]:
# Predicción: ¿qué calcula cada línea? (pista: True vale 1)
print(mascara.sum())
print(mascara.mean())

In [ ]:
df[mascara]

In [ ]:
# Varias condiciones: &, |, ~  y cada condición entre paréntesis
df[(df['ramo'] == 'GMM') & (df['siniestralidad'] > 1)]

In [ ]:
# Equivalente a IN de SQL o %in% de R
df[df['estado'].isin(['CDMX', 'Jalisco'])]

In [ ]:
# Filtrar filas y elegir columnas en un solo paso
df.loc[df['edad'] >= 40, ['ramo', 'prima', 'siniestralidad']]

### Modificar solo las filas que cumplen una condición
`loc` también sirve para **asignar**. Es el equivalente de un `UPDATE ... WHERE`:

In [ ]:
df['revision'] = 'No'
df.loc[df['siniestralidad'] > 1, 'revision'] = 'Sí'
df[['ramo', 'siniestralidad', 'revision']]

> **Atención:** no uses `df['revision'][mascara] = 'Sí'` (la "asignación encadenada"). Según la versión de pandas da una advertencia o simplemente no modifica `df`. Usa siempre `df.loc[mascara, 'columna'] = valor`.

> **Equivalencias**
>
> | pandas | R | SQL |
> |---|---|---|
> | `df[df['ramo'] == 'GMM']` | `filter(df, ramo == "GMM")` | `WHERE ramo = 'GMM'` |
> | `(a) & (b)`, `(a) \| (b)`, `~a` | `a & b`, `a \| b`, `!a` | `AND`, `OR`, `NOT` |
> | `df['estado'].isin([...])` | `estado %in% c(...)` | `estado IN (...)` |
> | `df.loc[cond, ['ramo', 'prima']]` | `df[cond, c("ramo", "prima")]` | `SELECT ramo, prima ... WHERE cond` |
> | `df.loc[cond, 'col'] = v` | `df$col[cond] <- v` | `UPDATE ... SET col = v WHERE cond` |

### Error a propósito
Quien viene de SQL suele escribir `and`. Ejecuta la celda y lee el mensaje: lo vas a ver muchas veces. ¿Cómo se corrige?

In [ ]:
df[df['ramo'] == 'GMM' and df['siniestralidad'] > 1]

### Ejercicio
1. Las pólizas de Guanajuato.
2. Las pólizas de Auto o Vida (usa `isin`).
3. Con `loc`: el estado de las pólizas **sin** siniestros.

In [ ]:
# 1
print(df[___])

# 2
print(df[___])

# 3
print(df.loc[___, ___])

In [ ]:
#@title Ver solución (doble clic para abrir)
print(df[df['estado'] == 'Guanajuato'])
print(df[df['ramo'].isin(['Auto', 'Vida'])])
print(df.loc[df['siniestros'] == 0, 'estado'])

---
## 5. Reto final

Con **un solo** `loc` y filtro booleano, muestra `ramo`, `prima` y `siniestralidad` de las pólizas de **Auto o GMM** con **siniestralidad mayor a 100 %**.

In [ ]:
df.loc[___, ___]

In [ ]:
#@title Ver solución (doble clic para abrir)
df.loc[df['ramo'].isin(['Auto', 'GMM']) & (df['siniestralidad'] > 1),
       ['ramo', 'prima', 'siniestralidad']]

### Extra para quien termine antes
¿Qué **porcentaje de la prima total** de la cartera está en pólizas con siniestralidad mayor a 100 %?

In [ ]:
pct = ___
print(f'{pct:.1%}')

In [ ]:
#@title Ver solución (doble clic para abrir)
pct = df.loc[df['siniestralidad'] > 1, 'prima'].sum() / df['prima'].sum()
print(f'{pct:.1%}')

---
### Resumen
- **Series** = arreglo de NumPy (operaciones vectorizadas) + diccionario (acceso y **alineación** por etiqueta).
- **DataFrame** = diccionario de Series que comparten el mismo índice.
- **`loc`** usa etiquetas e incluye el final al rebanar; **`iloc`** usa posiciones desde 0 y excluye el final.
- **Filtrado booleano**: `df[cond]` o `df.loc[cond, columnas]`, combinando con `&`, `|`, `~` y paréntesis. Para modificar: `df.loc[cond, col] = valor`.